#Bronze Layer

In [0]:
%sql
select * from samples.nyctaxi.trips limit 100

In [0]:
%sql
create table if not exists bronze_taxi as select * from samples.nyctaxi.trips

In [0]:
%sql
select * from bronze_taxi

In [0]:
%sql
DESCRIBE HISTORY bronze_taxi

#Silver Layer

In [0]:
from pyspark.sql.functions import col, unix_timestamp, round as spark_round

bronze_df = spark.table("bronze_taxi")

silver_df = (
    bronze_df
    .filter(col("fare_amount") > 0)
    .filter(col("trip_distance") > 0)
    .withColumn(
        "trip_duration_minutes",
        spark_round(
            (unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 60, 2
        )
    )
    .filter(col("trip_duration_minutes") > 0)  # drop bad timestamp rows
    .dropDuplicates()
)

silver_df.write.format("delta").mode("overwrite").saveAsTable("silver_taxi")

display(silver_df.limit(10))

#Gold Layer

In [0]:
%sql
CREATE OR REPLACE TABLE gold_taxi_by_zip AS
SELECT
  pickup_zip,
  COUNT(*) AS trip_count,
  ROUND(AVG(fare_amount), 2) AS avg_fare,
  ROUND(AVG(trip_distance), 2) AS avg_distance_miles,
  ROUND(AVG(trip_duration_minutes), 2) AS avg_duration_minutes
FROM silver_taxi
GROUP BY pickup_zip
HAVING COUNT(*) >= 20    -- drop zips with too few trips to trust the average
ORDER BY trip_count DESC;

CREATE OR REPLACE TABLE gold_taxi_by_hour AS
SELECT
  HOUR(tpep_pickup_datetime) AS pickup_hour,
  COUNT(*) AS trip_count,
  ROUND(AVG(fare_amount), 2) AS avg_fare
FROM silver_taxi
GROUP BY HOUR(tpep_pickup_datetime)
ORDER BY pickup_hour;

In [0]:
%sql
select * from gold_taxi_by_zip

In [0]:
%sql
select * from gold_taxi_by_hour

In [0]:
%sql
SELECT * FROM gold_taxi_by_zip ORDER BY avg_fare DESC LIMIT 10;

SQL Queries

In [0]:
%sql
select * from gold_taxi_by_zip order by trip_count desc limit 20